# 图像编码 Demo（教师版）

用于生成课堂 PPTX 所需素材、现场演示和 fallback 图片。请从本课目录运行本 notebook。

## 0. 工作目录检查

In [ ]:
from pathlib import Path

lesson_dir = Path.cwd()
assets_dir = lesson_dir / "assets"

assert assets_dir.is_dir(), (
    "当前 Jupyter working directory 不是本课目录："
    f"{lesson_dir}\n"
    "请从 1-2-encoding/1-2-3-image-encoding 目录启动 notebook。"
)

print("lesson directory:", lesson_dir)
print("assets directory:", assets_dir)


## 1. 生成固定测试图

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw

path = Path("assets/test-image.png")
path.parent.mkdir(parents=True, exist_ok=True)

w = h = 256
img = Image.new("RGB", (w, h))
px = img.load()

# 水平灰度渐变：用于观察 banding
for y in range(h):
    for x in range(w):
        v = round(255 * x / (w - 1))
        px[x, y] = (v, v, v)

draw = ImageDraw.Draw(img)

# 红、绿两个色块直接相接：接缝位于 x=72 / 73；蓝色作参照
draw.rectangle((12, 12, 72, 72), fill=(255, 0, 0))
draw.rectangle((73, 12, 133, 72), fill=(0, 255, 0))
draw.rectangle((184, 12, 244, 72), fill=(0, 0, 255))

# 下半部浅灰背景 + 大圆 + 粗斜线 + 细节条纹
draw.rectangle((0, 104, 128, 255), fill=(224, 224, 224))
draw.rectangle((136, 104, 255, 255), fill=(224, 224, 224))
draw.ellipse((0, 112, 128, 240), outline=(0, 0, 0), width=28)
draw.line((144, 240, 240, 144), fill=(0, 0, 0), width=28)
for x in [152, 160, 168, 176, 184, 192]:
    draw.line((x, 224, x, 244), fill=(255, 255, 255), width=2)

img.save(path)

assert img.size == (256, 256)
assert img.mode == "RGB"
assert path.is_file()

print("created:", path)


## 2. Demo A：Sampling / Resolution

主课默认使用 PPTX 两张证据页。本 demo 主要用于生成 fallback，或在学生提出新尺寸时现场验证。

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

source_path = Path("assets/test-image.png")
assert source_path.is_file(), "请先运行固定测试图生成单元。"

src = Image.open(source_path).convert("RGB")
sizes = [256, 64, 32, 16, 8]

sampling_outputs = {}

for n in sizes:
    sampled = src.resize((n, n), resample=Image.Resampling.NEAREST)
    enlarged = sampled.resize((512, 512), resample=Image.Resampling.NEAREST)

    assert sampled.mode == "RGB"
    assert enlarged.mode == "RGB"
    assert enlarged.size == (512, 512)

    sampling_outputs[n] = enlarged

    print(f"{n} × {n}")
    display(enlarged)


## 3. Demo B：Quantization

先展示图像，再揭示接缝数值。

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

source_path = Path("assets/test-image.png")
assert source_path.is_file()

src = Image.open(source_path).convert("RGB")
targets = [256, 16, 4, 2]
main_targets = [16, 2]
outputs = {}

for n in targets:
    q = src.quantize(
        colors=n,
        method=Image.Quantize.MEDIANCUT,
        dither=Image.Dither.NONE,
    ).convert("RGB")

    colors = q.getcolors(maxcolors=n + 1)
    assert colors is not None
    color_count = len(colors)

    assert q.size == src.size
    assert q.mode == "RGB"
    assert color_count <= n

    outputs[n] = q

print("original")
display(src)

for n in main_targets:
    print(f"max {n} colors")
    display(outputs[n])

# 256-color / 4-color 已保存在 outputs 中。需要追问时再 display(outputs[256]) / display(outputs[4])。


In [ ]:
seam_left = (72, 40)
seam_right = (73, 40)

for n in [16, 2]:
    left_rgb = outputs[n].getpixel(seam_left)
    right_rgb = outputs[n].getpixel(seam_right)

    print(
        f"{n:2d} colors | "
        f"left={left_rgb} | right={right_rgb} | "
        f"same={left_rgb == right_rgb}"
    )


## 4. 导出并复核 fallback 图片

In [ ]:
from pathlib import Path
from PIL import Image

fallback_dir = Path("assets/fallback")
fallback_dir.mkdir(parents=True, exist_ok=True)

# Demo A：保存课堂实际展示的五张 512×512 图
for n in [256, 64, 32, 16, 8]:
    img = sampling_outputs[n]
    assert img.mode == "RGB"
    assert img.size == (512, 512)

    path = fallback_dir / f"sampling-{n}.png"
    img.save(path)

# Demo B：保存主课三张量化证据图
quant_images = {
    "original": src,
    "16": outputs[16],
    "2": outputs[2],
}

for label, img in quant_images.items():
    assert img.mode == "RGB"
    assert img.size == (256, 256)

    path = fallback_dir / f"quant-{label}.png"
    img.save(path)

print("fallback images exported to:", fallback_dir)


In [ ]:
from PIL import Image

# 1) Demo A：五张图都应是 512×512 RGB
for n in [256, 64, 32, 16, 8]:
    path = fallback_dir / f"sampling-{n}.png"
    with Image.open(path) as img:
        assert img.mode == "RGB"
        assert img.size == (512, 512)

# 2) Demo B：三张图都应是 256×256 RGB
for label in ["original", "16", "2"]:
    path = fallback_dir / f"quant-{label}.png"
    with Image.open(path) as img:
        assert img.mode == "RGB"
        assert img.size == (256, 256)

# 3) 红绿接缝：用“刚导出的文件”再检查一次
seam_left = (72, 40)
seam_right = (73, 40)

with Image.open(fallback_dir / "quant-16.png").convert("RGB") as q16:
    assert q16.getpixel(seam_left) != q16.getpixel(seam_right)

with Image.open(fallback_dir / "quant-2.png").convert("RGB") as q2:
    assert q2.getpixel(seam_left) == q2.getpixel(seam_right)

print("fallback verification passed")


## 5. Demo C：一个 RGB 像素到 bits

In [ ]:
def byte_bits(x):
    return format(x, "08b")

rgb = (255, 128, 0)
print("RGB =", rgb)
print("bits:", " ".join(byte_bits(x) for x in rgb))


## 6. Demo D：BMP 文件大小与 row padding（拓展）

In [ ]:
from pathlib import Path
import struct

path = Path("assets/q7-3x2-24bit.bmp")
assert path.is_file(), f"找不到 {path}。请确认 Jupyter 从本课目录启动。"

data = path.read_bytes()

pixel_offset = struct.unpack_from("<I", data, 10)[0]
width = struct.unpack_from("<i", data, 18)[0]
height = struct.unpack_from("<i", data, 22)[0]
bits_per_pixel = struct.unpack_from("<H", data, 28)[0]
rows = abs(height)

color_value_bytes = width * rows * bits_per_pixel // 8
row_value_bytes = width * bits_per_pixel // 8
stored_row_bytes = ((width * bits_per_pixel + 31) // 32) * 4
pixel_array_bytes = stored_row_bytes * rows
file_bytes = len(data)

print("pixel-data offset :", pixel_offset, "bytes")
print("color values      :", color_value_bytes, "bytes")
print("raw row values    :", row_value_bytes, "bytes/row")
print("stored row size   :", stored_row_bytes, "bytes/row")
print("pixel array       :", pixel_array_bytes, "bytes")
print("whole file        :", file_bytes, "bytes")

assert pixel_offset == 54
assert color_value_bytes == 18
assert stored_row_bytes == 12
assert pixel_array_bytes == 24
assert file_bytes == 78


## 7. 可选：验证 24-bit BMP 文件层证据（拓展）

In [ ]:
from pathlib import Path
import struct

out_dir = Path("demo-output/quantization-bmp")
out_dir.mkdir(parents=True, exist_ok=True)

for n, q in outputs.items():
    bmp_path = out_dir / f"max-{n}-colors.bmp"
    q.save(bmp_path, format="BMP")

    data = bmp_path.read_bytes()
    bits_per_pixel = struct.unpack_from("<H", data, 28)[0]

    print(
        bmp_path.name,
        "| bpp =", bits_per_pixel,
        "| file bytes =", len(data),
    )
